In [9]:
import os
import glob
import h5py
import numpy as np
import scipy.signal
import pandas as pd

# 1. Update folder path if needed
file_pattern = r"/mnt/d/GCS(trial)/data/chime-frb-open-data-master/files_images/*.h5" 
file_list = glob.glob(file_pattern)

print(f"Detected {len(file_list)} files. Starting native extraction pipeline...\n")

def boxcar_kernel(width):
    width = int(round(width, 0))
    return np.ones(width, dtype="float32") / np.sqrt(width)

def find_burst(ts, min_width=1, max_width=128):
    min_width = int(min_width)
    max_width = int(max_width)
    widths = list(range(min_width, min(max_width + 1, len(ts)-2)))
    
    # Fill NaNs with 0.0 to prevent convolve issues
    ts_clean = np.nan_to_num(ts, nan=0.0)
    
    snrs = np.empty_like(widths, dtype=float)
    peaks = np.empty_like(widths, dtype=int)
    
    for i in range(len(widths)):
        convolved = scipy.signal.convolve(ts_clean, boxcar_kernel(widths[i]), mode="same")
        peaks[i] = np.nanargmax(convolved)
        snrs[i] = convolved[peaks[i]]
        
    best_idx = np.nanargmax(snrs)
    return peaks[best_idx], widths[best_idx], snrs[best_idx]

# List to gather rows for our final table
table_data = []

# 2. Main processing loop
for path2 in file_list:
    try:
        with h5py.File(path2, 'r') as data:
            frb_group = data['frb']
            eventname = frb_group.attrs['tns_name'].decode()
            
            wfall = frb_group['wfall'][:]
            plot_time = frb_group["plot_time"][:]
            spec = frb_group['spec'][:]
            dm = frb_group.attrs['dm'][()]
            
        q1 = np.nanquantile(spec, 0.25)
        q3 = np.nanquantile(spec, 0.75)
        iqr = q3 - q1

        rfi_masking_var_factor = 3
        channel_variance = np.nanvar(wfall, axis=1)
        mean_channel_variance = np.nanmean(channel_variance)

        with np.errstate(invalid="ignore"):
            rfi_mask = (
                (channel_variance > rfi_masking_var_factor * mean_channel_variance) | 
                (spec[::-1] < q1 - 1.5 * iqr) | 
                (spec[::-1] > q3 + 1.5 * iqr)
            )
                       
        wfall[rfi_mask,...] = np.nan
        ts = np.nansum(wfall, axis=0)
        
        # Calculate time resolution (sampling time)
        dt = np.median(np.diff(plot_time))

        # Run burst profiling
        peak_idx, optimal_width, max_snr = find_burst(ts)
        
        # Calculate physical width in milliseconds (dt is in seconds)
        width_ms = float(optimal_width) * float(dt) * 1000.0
        
        # Append elements with full precision (no rounding)
        table_data.append({
            "Event Name": eventname,
            "File Name": os.path.basename(path2),
            "DM": float(dm),
            "SNR": float(max_snr),
            "Width (bins)": int(optimal_width),
            "Width (ms)": float(width_ms),
            "Peak Index": int(peak_idx)
        })
        
    except Exception as e:
        print(f"Could not process {os.path.basename(path2)} due to: {e}")

# 3. Render compiled results as a neat structured dataframe 
if table_data:
    df_results = pd.DataFrame(table_data)
    display(df_results)
else:
    print("No event records were generated. Ensure file paths map to real data.")


Detected 10 files. Starting native extraction pipeline...



/tmp/ipykernel_29787/2782442776.py:57: RuntimeWarning: Degrees of freedom <= 0 for slice.
  channel_variance = np.nanvar(wfall, axis=1)


,Event Name,File Name,DM,SNR,Width (bins),Width (ms),Peak Index
0,FRB20180725A,FRB20180725A_waterfall.h5,715.809266,311.775313,3,2949.119982,13
1,FRB20180727A,FRB20180727A_waterfall.h5,642.133745,148.256974,3,2949.119982,12
2,FRB20180730A,FRB20180730A_waterfall.h5,848.904087,1039.461321,5,4915.199825,14
3,FRB20180801A,FRB20180801A_waterfall.h5,655.728021,354.825351,10,9830.399940,29
4,FRB20180806A,FRB20180806A_waterfall.h5,739.948240,208.396786,3,2949.119895,12
5,FRB20180812A,FRB20180812A_waterfall.h5,802.450582,179.913594,8,7864.320010,23
6,FRB20180814A,FRB20180814A_waterfall.h5,189.285546,136.173735,8,7864.319952,13
7,FRB20180817A,FRB20180817A_waterfall.h5,1006.771381,762.669553,18,17694.719369,48
8,FRB20180904A,FRB20180904A_waterfall.h5,361.137415,565.499184,1,983.039994,6
9,FRB20190701D,FRB20190701D_waterfall.h5,933.362937,330.434750,10,9830.399649,16


Width (ms) Column: Dynamically multiplies the optimal width bin value by dt and scales it by 1000.0 (converting from seconds to milliseconds).

Unrounded Metrics: Retains full precision data values across all statistical calculations.

In [1]:
import os
import glob
import h5py
import numpy as np
import scipy.signal
import pandas as pd

# 1. Update folder path if needed
file_pattern = r"/mnt/d/GCS(trial)/data/chime-frb-open-data-master/files_images/*.h5" 
file_list = glob.glob(file_pattern)

print(f"Detected {len(file_list)} files. Starting native extraction pipeline...\n")

def boxcar_kernel(width):
    width = int(round(width, 0))
    return np.ones(width, dtype="float32") / np.sqrt(width)

def find_burst(ts, min_width=1, max_width=128):
    min_width = int(min_width)
    max_width = int(max_width)
    widths = list(range(min_width, min(max_width + 1, len(ts)-2)))
    
    # Fill NaNs with 0.0 to prevent convolve issues
    ts_clean = np.nan_to_num(ts, nan=0.0)
    
    snrs = np.empty_like(widths, dtype=float)
    peaks = np.empty_like(widths, dtype=int)
    
    for i in range(len(widths)):
        convolved = scipy.signal.convolve(ts_clean, boxcar_kernel(widths[i]), mode="same")
        peaks[i] = np.nanargmax(convolved)
        snrs[i] = convolved[peaks[i]]
        
    best_idx = np.nanargmax(snrs)
    return peaks[best_idx], widths[best_idx], snrs[best_idx]

# List to gather rows for our final table
table_data = []

# 2. Main processing loop
for path2 in file_list:
    try:
        with h5py.File(path2, 'r') as data:
            frb_group = data['frb']
            eventname = frb_group.attrs['tns_name'].decode()
            
            wfall = frb_group['wfall'][:]
            plot_time = frb_group["plot_time"][:]
            spec = frb_group['spec'][:]
            dm = frb_group.attrs['dm'][()]
            
        q1 = np.nanquantile(spec, 0.25)
        q3 = np.nanquantile(spec, 0.75)
        iqr = q3 - q1

        rfi_masking_var_factor = 3
        channel_variance = np.nanvar(wfall, axis=1)
        mean_channel_variance = np.nanmean(channel_variance)

        with np.errstate(invalid="ignore"):
            rfi_mask = (
                (channel_variance > rfi_masking_var_factor * mean_channel_variance) | 
                (spec[::-1] < q1 - 1.5 * iqr) | 
                (spec[::-1] > q3 + 1.5 * iqr)
            )
                       
        wfall[rfi_mask,...] = np.nan
        ts = np.nansum(wfall, axis=0)
        
        # Calculate time resolution (sampling time)
        dt = np.median(np.diff(plot_time))

        # Run burst profiling
        peak_idx, optimal_width, max_snr = find_burst(ts)
        
        # Calculate physical width in milliseconds (dt is in seconds)
        width_ms = float(optimal_width) * float(dt) * 1000.0
        
        # Append elements with full precision (no rounding)
        table_data.append({
            "Event Name": eventname,
            "File Name": os.path.basename(path2),
            "DM": float(dm),
            "SNR": float(max_snr),
            "Width (bins)": int(optimal_width),
            "Width (ms)": float(width_ms),
            "Peak Index": int(peak_idx)
        })
        
    except Exception as e:
        print(f"Could not process {os.path.basename(path2)} due to: {e}")

# =====================================================================
# 3. Render compiled results as a dataframe AND save to CSV file
# =====================================================================
if table_data:
    df_results = pd.DataFrame(table_data)
    
    # Define file storage location destination path string variables
    output_csv_filename = "frb_pipeline_results.csv"
    
    # Commit entire compiled target matrix data frames down to local system disk pathing
    df_results.to_csv(output_csv_filename, index=False)
    print(f"\n--> Successfully saved tracking records to output file: '{output_csv_filename}'")
    
    # Present interactive visualization outputs safely inside the notebook session engine
    display(df_results)
else:
    print("No event records were generated. Ensure file paths map to real data.")


Detected 30 files. Starting native extraction pipeline...



/tmp/ipykernel_122572/3053365199.py:57: RuntimeWarning: Degrees of freedom <= 0 for slice.
  channel_variance = np.nanvar(wfall, axis=1)



--> Successfully saved tracking records to output file: 'frb_pipeline_results.csv'


,Event Name,File Name,DM,SNR,Width (bins),Width (ms),Peak Index
0,FRB20180725A,FRB20180725A_waterfall.h5,715.809266,311.775313,3,2949.119982,13
1,FRB20180727A,FRB20180727A_waterfall.h5,642.133745,148.256974,3,2949.119982,12
2,FRB20180730A,FRB20180730A_waterfall.h5,848.904087,1039.461321,5,4915.199825,14
3,FRB20180801A,FRB20180801A_waterfall.h5,655.728021,354.825351,10,9830.399940,29
4,FRB20180806A,FRB20180806A_waterfall.h5,739.948240,208.396786,3,2949.119895,12
5,FRB20180812A,FRB20180812A_waterfall.h5,802.450582,179.913594,8,7864.320010,23
6,FRB20180814A,FRB20180814A_waterfall.h5,189.285546,136.173735,8,7864.319952,13
7,FRB20180817A,FRB20180817A_waterfall.h5,1006.771381,762.669553,18,17694.719369,48
8,FRB20180904A,FRB20180904A_waterfall.h5,361.137415,565.499184,1,983.039994,6
9,FRB20180909A,FRB20180909A_waterfall.h5,408.647151,93.658301,5,19660.799881,9
